# Certify-or-Grow vs. Nemecek & Parr's Policy Caches (ICML 2021), head-to-head on DST

[Nemecek & Parr, "Policy Caches with Successor Features"](http://proceedings.mlr.press/v139/nemecek21a/nemecek21a.pdf)
(ICML 2021) address the same question as Certify-or-Grow -- *given a cache of base
policies and a new query weight, is the cache good enough, or do we need a new exact
solve?* -- but via a different certificate.

**Their Algorithm 1 (`LearnNewPolicy?`)**: compute an upper bound `ub` on the new task's
optimal value via a cheap LP over cached policies' own values (their Theorem 1 /
Algorithm 2, a generalization of Hunt et al.'s bound to hard-max Q-functions and conical
combinations), a lower bound `lb` via Barreto et al.'s standard GPI estimate, and grow
the cache iff `(ub - lb)/|lb| > eta` for a chosen threshold `eta`. If not, they deploy
**GPI** over the cached pool.

**Certify-or-Grow**, by contrast, runs restricted policy iteration to convergence and
checks the TRUE Bellman residual (Definition 3 in `notes/adaptive_basis.tex`) -- an
*exact* test with no hyperparameter, deploying the **metapolicy** when certified.

This notebook faithfully reproduces both algorithms (their Theorem 1 upper bound via the
literal LP of their Algorithm 2, not a shortcut) and compares them on the same $k=2$ base
pool and the same stream of 150 random query weights on DST.

## 1. DST's exact MDP (BFS-discovered) and exact solvers

In [1]:
import numpy as np
import mo_gymnasium as mo_gym
from scipy.optimize import linprog

GAMMA = 0.98
N_ACTIONS = 4


def make_env():
    return mo_gym.make("deep-sea-treasure-v0")


def discover_mdp(max_states=2000):
    """BFS-discover DST's full deterministic transition graph and rewards."""
    env = make_env()
    obs, _ = env.reset(seed=0)
    start = tuple(int(x) for x in obs)
    trans = {}
    frontier = [start]
    seen = {start}
    while frontier and len(seen) < max_states:
        s = frontier.pop()
        for a in range(N_ACTIONS):
            e2 = make_env()
            e2.reset(seed=0)
            e2.unwrapped.current_state = np.array(s)
            obs2, r, term, trunc, _ = e2.step(a)
            s2 = tuple(int(x) for x in obs2)
            trans[(s, a)] = (s2, np.array(r, dtype=float), term)
            if s2 not in seen:
                seen.add(s2)
                frontier.append(s2)
    return start, list(seen), trans


def policy_iteration(states, trans, w, gamma, allowed=None):
    idx = {s: i for i, s in enumerate(states)}
    n = len(states)
    allowed = allowed or {s: list(range(N_ACTIONS)) for s in states}
    pi = {s: allowed[s][0] for s in states}
    for _ in range(1000):
        A, b = np.eye(n), np.zeros(n)
        for s in states:
            i = idx[s]
            s2, r, term = trans[(s, pi[s])]
            b[i] = r @ w
            if not term:
                A[i, idx[s2]] -= gamma
        V = np.linalg.solve(A, b)
        stable = True
        for s in states:
            qs = []
            for a in allowed[s]:
                s2, r, term = trans[(s, a)]
                qs.append(r @ w + (0.0 if term else gamma * V[idx[s2]]))
            best_a = allowed[s][int(np.argmax(qs))]
            stable &= (best_a == pi[s])
            pi[s] = best_a
        if stable:
            break
    return pi, {s: V[idx[s]] for s in states}


def full_sf_table(states, trans, pi, gamma, feat_dim=2):
    idx = {s: i for i, s in enumerate(states)}
    n = len(states)
    A, b = np.eye(n), np.zeros((n, feat_dim))
    for s in states:
        i = idx[s]
        s2, r, term = trans[(s, pi[s])]
        b[i] = r
        if not term:
            A[i, idx[s2]] -= gamma
    Vvec = np.linalg.solve(A, b)
    Vvec_d = {s: Vvec[idx[s]] for s in states}
    Psi = {}
    for s in states:
        row = np.zeros((N_ACTIONS, feat_dim))
        for a in range(N_ACTIONS):
            s2, r, term = trans[(s, a)]
            row[a] = r if term else r + gamma * Vvec_d[s2]
        Psi[s] = row
    return Psi, Vvec_d


def bellman_gap(states, trans, w, gamma, V):
    worst = 0.0
    for s in states:
        qs = []
        for a in range(N_ACTIONS):
            s2, r, term = trans[(s, a)]
            qs.append(r @ w + (0.0 if term else gamma * V[s2]))
        worst = max(worst, max(qs) - V[s])
    return worst


def gpi_policy_and_value(states, trans, w, gamma, Psi_list):
    idx = {s: i for i, s in enumerate(states)}
    n = len(states)
    pi = {}
    for s in states:
        best_a, best_q = 0, -np.inf
        for a in range(N_ACTIONS):
            q = max(Psi[s][a] @ w for Psi in Psi_list)
            if q > best_q:
                best_q, best_a = q, a
        pi[s] = best_a
    A, b = np.eye(n), np.zeros(n)
    for s in states:
        i = idx[s]
        s2, r, term = trans[(s, pi[s])]
        b[i] = r @ w
        if not term:
            A[i, idx[s2]] -= gamma
    V = np.linalg.solve(A, b)
    return pi, {s: V[idx[s]] for s in states}


def np_upper_bound_lp(V_bases, W_bases, w_new):
    """Nemecek & Parr's Algorithm 2 (CalcUpperBound): the LP behind Theorem 1's upper
    bound, minimize sum(alpha_i * V_i(s)) s.t. sum(alpha_i * W_i) = w_new, alpha_i>=0."""
    m = len(V_bases)
    c = np.array(V_bases)
    A_eq = np.array(W_bases).T
    b_eq = np.array(w_new)
    res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=[(0, None)] * m, method="highs")
    if not res.success:
        raise RuntimeError("LP infeasible: w_new not in the conical hull of base weights")
    return res.fun


## 2. The $k=2$ base pool (canonical corner weights), solved exactly

In [2]:
start, states, trans = discover_mdp()
print(f"DST: {len(states)} reachable states discovered via BFS.")
gamma = GAMMA

w0, w1 = np.array([1.0, 0.0]), np.array([0.0, 1.0])
pi0, V0 = policy_iteration(states, trans, w0, gamma)
pi1, V1 = policy_iteration(states, trans, w1, gamma)
Psi0, _ = full_sf_table(states, trans, pi0, gamma)
Psi1, _ = full_sf_table(states, trans, pi1, gamma)
V0_self, V1_self = V0[start], V1[start]
print(f"Base pool (k=2): pi_0 (w=(1,0)), V_0(s0)={V0_self:.4f};"
      f"  pi_1 (w=(0,1)), V_1(s0)={V1_self:.4f}")


/opt/anaconda3/envs/okb/lib/python3.10/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


DST: 72 reachable states discovered via BFS.


Base pool (k=2): pi_0 (w=(1,0)), V_0(s0)=16.4747;  pi_1 (w=(0,1)), V_1(s0)=-1.0000


## 3. 150 random query weights: certify-or-grow vs. Nemecek & Parr at five thresholds

For each query weight, ground truth ($V^*_w(s_0)$) comes from exact unrestricted policy
iteration. Certify-or-Grow's realized suboptimality is measured against its own
(restricted) metapolicy value when certified; Nemecek & Parr's against **GPI's** realized
value when their bound says the cache suffices -- each method's own natural deployment
choice, matching both papers' actual proposals.

In [3]:
rng = np.random.default_rng(0)
n_weights = 150
query_weights = [rng.dirichlet(np.ones(2)) for _ in range(n_weights)]

etas = [0.01, 0.05, 0.1, 0.2, 0.3]
np_stats = {eta: {"n_sufficient": 0, "subopt": []} for eta in etas}
cog_stats = {"n_certified": 0, "subopt": []}

for w in query_weights:
    _, V_star = policy_iteration(states, trans, w, gamma)
    vstar_s0 = V_star[start]

    # --- Certify-or-Grow: exact Bellman-gap test on the restricted metapolicy ---
    allowed = {s: sorted(set([pi0[s], pi1[s]])) for s in states}
    pi_meta, V_meta = policy_iteration(states, trans, w, gamma, allowed=allowed)
    gap = bellman_gap(states, trans, w, gamma, V_meta)
    if gap < 1e-6:
        cog_stats["n_certified"] += 1
        cog_stats["subopt"].append(vstar_s0 - V_meta[start])

    # --- Nemecek & Parr: Theorem 1 bound sandwich, thresholded by eta (Algorithm 1) ---
    ub = np_upper_bound_lp([V0_self, V1_self], [w0, w1], w)
    lb = max(max(Psi0[start][a] @ w for a in range(N_ACTIONS)),
             max(Psi1[start][a] @ w for a in range(N_ACTIONS)))
    ratio = (ub - lb) / abs(lb) if lb != 0 else np.inf
    _, V_gpi = gpi_policy_and_value(states, trans, w, gamma, [Psi0, Psi1])
    real_subopt = vstar_s0 - V_gpi[start]
    for eta in etas:
        if ratio <= eta:
            np_stats[eta]["n_sufficient"] += 1
            np_stats[eta]["subopt"].append(real_subopt)

print(f"Certify-or-Grow: certified {cog_stats['n_certified']}/{n_weights} query weights "
      f"({100*cog_stats['n_certified']/n_weights:.1f}%)")
if cog_stats["subopt"]:
    print(f"  realized suboptimality when certified: mean={np.mean(cog_stats['subopt']):.6f}, "
          f"max={np.max(cog_stats['subopt']):.6f}  (exact certificate -- should be ~0)")
print()
print("Nemecek & Parr (Algorithm 1), by threshold eta -- realized policy is GPI over the pool:")
print(f"{'eta':>6} {'n_sufficient':>13} {'%':>7} {'mean subopt':>13} {'max subopt':>12}")
for eta in etas:
    st = np_stats[eta]
    n = st["n_sufficient"]
    mean_s = np.mean(st["subopt"]) if st["subopt"] else float("nan")
    max_s = np.max(st["subopt"]) if st["subopt"] else float("nan")
    print(f"{eta:>6} {n:>13} {100*n/n_weights:>6.1f}% {mean_s:>13.6f} {max_s:>12.6f}")


Certify-or-Grow: certified 6/150 query weights (4.0%)
  realized suboptimality when certified: mean=0.000000, max=0.000000  (exact certificate -- should be ~0)

Nemecek & Parr (Algorithm 1), by threshold eta -- realized policy is GPI over the pool:
   eta  n_sufficient       %   mean subopt   max subopt
  0.01             2    1.3%      0.000000     0.000000
  0.05             6    4.0%      0.000000     0.000000
   0.1            14    9.3%      0.000000     0.000000
   0.2            24   16.0%      0.000000     0.000000
   0.3            37   24.7%      0.002503     0.052429


## Conclusion

Certify-or-Grow's exact zero-gap certificate accepts the $k=2$ cache for 4.0% of query
weights, always with exactly zero realized suboptimality -- true by construction, and
confirmed numerically here. Nemecek & Parr's cheaper LP-bound certificate is *more
permissive* at moderate thresholds (up to 16.0% of weights at $\eta=0.2$) while still
measuring exactly zero suboptimality -- their bound is tight enough in this domain that a
reasonably generous threshold costs nothing. Only at a looser threshold ($\eta=0.3$,
24.7% accepted) does the bound's slack let real suboptimality through (mean $0.0025$, max
$0.0524$). The natural reading: Nemecek & Parr's decision targets GPI's generally-better
realized quality (so a wider acceptance region stays safe), while Certify-or-Grow's more
restrictive metapolicy needs the true optimum to live entirely inside $B(s)$ at every
visited state -- a strictly rarer event -- but is exact with no threshold to tune. Neither
dominates: one trades a hyperparameter and occasional (bounded, here small) slack for a
cheap LP and wider cache reuse; the other trades a costlier per-query restricted-PI solve
for a hyperparameter-free, always-exact guarantee.